# Lab 5 - Config
Shared settings for all lab5 notebooks. Picks the catalog, schema, secret scope and paths for the chosen environment (`personal`, `azure_dev`, `azure_trial` or `azure_trial_2`), adds the shared code from `src/lab5` to the Python path and defines helpers for table names, Delta versions and DataFrame comparison.

In [ ]:
import re
import sys
from pathlib import Path
from pyspark.sql import functions as F


dbutils.widgets.dropdown("environment", "personal", ["personal", "azure_dev", "azure_trial", "azure_trial_2"])
for name in ["catalog", "schema", "source_csv_path", "source_code_path", "secret_scope"]:
    dbutils.widgets.text(name, "")
dbutils.widgets.text("eventhub_namespace", "evhpl24databricks02")
dbutils.widgets.text("eventhub_name", "ivanrazumovskyi_evh")
dbutils.widgets.text("secret_key", "eventhub-connection-string")

environment = dbutils.widgets.get("environment")
catalog = dbutils.widgets.get("catalog") or ({"personal": "workspace", "azure_dev": "dbr_dev", "azure_trial": "dbr_dev_trial", "azure_trial_2": "dbr_dev"}[environment])
schema = dbutils.widgets.get("schema") or "ivanrazumovskyi_lab5"
for identifier in [catalog, schema]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError("Use simple catalog/schema identifiers")
namespace = dbutils.widgets.get("eventhub_namespace")
eh_name = dbutils.widgets.get("eventhub_name")
secret_scope = dbutils.widgets.get("secret_scope") or ("ivanrazumovskyi-kv-scope2" if environment == "azure_dev" else "lab5-eventhub")
secret_key = dbutils.widgets.get("secret_key")
volume_path = f"/Volumes/{catalog}/{schema}/lab5_sources"
source_csv_path = dbutils.widgets.get("source_csv_path") or f"{volume_path}/netflix/netflix_titles.csv"
source_code_path = dbutils.widgets.get("source_code_path")
if not source_code_path:
    candidates = [Path.cwd().parent / "src", Path.cwd() / "src"]
    source_code_path = next((str(p) for p in candidates if (p / "lab5" / "transforms.py").exists()), "")
if not source_code_path:
    raise ValueError("Set source_code_path to the deployed bundle's src directory")
sys.path.insert(0, source_code_path)
from lab5.contracts import TITLE_RULES, WIKI_RULES, TITLE_BUSINESS_COLUMNS, WIKI_BUSINESS_COLUMNS
from lab5.transforms import read_titles, clean_titles, parse_wiki, annotate_quality, valid_rows, kafka_options, decode_kafka

spark.conf.set("spark.sql.session.timeZone", "UTC")
def table(name):
    return f"{catalog}.{schema}.{name}_lab5"
def delta_version(name):
    return int(spark.sql(f"DESCRIBE HISTORY {table(name)} LIMIT 1").first()["version"])
def read_version(name, version):
    return spark.read.option("versionAsOf", version).table(table(name))
def assert_equal(actual, expected, columns, label):
    left, right = actual.select(*columns), expected.select(*columns)
    assert left.exceptAll(right).limit(1).count() == 0, f"{label}: unexpected rows"
    assert right.exceptAll(left).limit(1).count() == 0, f"{label}: missing rows"
    print(f"PASS: {label} (multiset equality)")
print({"environment": environment, "catalog": catalog, "schema": schema,
       "source_csv_path": source_csv_path, "source_code_path": source_code_path,
       "eventhub": f"{namespace}/{eh_name}"})